# Bond Wire Defect Types

Epic 6 implements three common bond wire defects found in AOI inspection:

1. **Bend (wire sweep)** -- lateral or vertical deformation
2. **Break** -- wire fracture creating two fragments
3. **Lift (lift-off)** -- pad detachment raising one end

Each defect is applied to a healthy `BondWireProfile` to produce a defective one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from udm_epic6.models.wire_geometry import (
    BondWireProfile, render_wire_mask,
    _bezier_control_points, _evaluate_bezier,
)
from udm_epic6.models.defect_generator import (
    apply_bend_defect, apply_break_defect, apply_lift_defect,
)

In [ ]:
# Healthy baseline wire
healthy = BondWireProfile(
    start_xy=(30, 128), end_xy=(226, 128),
    loop_height=35, diameter=3, curvature=0.0, material="gold",
)

rng = np.random.default_rng(42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def plot_wire(ax, profile, title, colour='blue'):
    cps = _bezier_control_points(profile)
    curve = _evaluate_bezier(cps)
    ax.plot(curve[:, 0], curve[:, 1], linewidth=profile.diameter, color=colour)
    ax.plot(*profile.start_xy, 'gs', markersize=8)
    ax.plot(*profile.end_xy, 'rs', markersize=8)
    ax.set_xlim(0, 256)
    ax.set_ylim(200, 50)
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')

# Healthy
plot_wire(axes[0, 0], healthy, "Healthy Wire")

# Bend defect
bent = apply_bend_defect(healthy, severity=0.7, rng=np.random.default_rng(42))
plot_wire(axes[0, 1], bent, "Bent Wire (severity=0.7)", colour='orange')

# Break defect
frag1, frag2 = apply_break_defect(healthy, break_position=0.4, rng=np.random.default_rng(42))
plot_wire(axes[1, 0], frag1, "Broken Wire - Fragment 1", colour='red')
cps2 = _bezier_control_points(frag2)
curve2 = _evaluate_bezier(cps2)
axes[1, 0].plot(curve2[:, 0], curve2[:, 1], linewidth=frag2.diameter, color='darkred')
axes[1, 0].set_title("Broken Wire (break_position=0.4)")

# Lift defect
lifted = apply_lift_defect(healthy, lift_amount=25.0, rng=np.random.default_rng(42))
plot_wire(axes[1, 1], lifted, "Lifted Wire (lift=25px)", colour='purple')

fig.tight_layout()
plt.show()

## Severity sweep

The bend defect accepts a `severity` parameter from 0.1 (mild) to 0.9 (extreme):

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for i, sev in enumerate([0.1, 0.3, 0.5, 0.7, 0.9]):
    bent = apply_bend_defect(healthy, severity=sev, rng=np.random.default_rng(i))
    mask = render_wire_mask(bent, 256, 256)
    axes[i].imshow(mask, cmap='hot')
    axes[i].set_title(f"severity={sev}")
    axes[i].axis('off')
fig.suptitle("Bend Defect Severity Sweep", fontsize=14)
fig.tight_layout()
plt.show()